This page upserts a few records into an in-memory SQLite database with each of the three strategies, so you can see in a minute what each one does. Install the library first:

```console
pip install sqlalchemy-upsert-kit
```

## A table and a database

Any SQLAlchemy Core table with one primary key column works. ORM models work too: pass `Model.__table__`.

In [1]:
import sqlalchemy as sa

from sqlalchemy_upsert_kit.api import sqlite

metadata = sa.MetaData()

t_user = sa.Table(
    "users",
    metadata,
    sa.Column("id", sa.Integer, primary_key=True),
    sa.Column("name", sa.String(50)),
    sa.Column("email", sa.String(50), unique=True),
)

engine = sa.create_engine("sqlite://")  # an in-memory SQLite database

Each example below starts from the same three rows, so here is a small function that resets the table to them.

In [2]:
existing_rows = [
    {"id": 1, "name": "alice", "email": "alice@old.com"},
    {"id": 2, "name": "bob", "email": "bob@old.com"},
    {"id": 3, "name": "cathy", "email": "cathy@old.com"},
]


def reset():
    """Recreate the table with the three existing rows."""
    metadata.drop_all(engine)
    metadata.create_all(engine)
    with engine.begin() as conn:
        conn.execute(t_user.insert(), existing_rows)

In [3]:
reset()
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@old.com'),
 (3, 'cathy', 'cathy@old.com')]

## The records to upsert

Users 2 and 3 already exist, and their records bring a new name and a new email; user 4 is new. Records are plain dicts keyed by column, and all of them have the same keys.

In [4]:
records = [
    {"id": 2, "name": "robert", "email": "bob@new.com"},
    {"id": 3, "name": "catherine", "email": "cathy@new.com"},
    {"id": 4, "name": "dave", "email": "dave@new.com"},
]

## Insert or ignore

Insert the new rows and leave the existing ones as they are. The function returns two counts: the records whose primary key already existed (here: ignored), and the rows inserted.

In [5]:
reset()
ignored, inserted = sqlite.insert_or_ignore(engine, t_user, records)
print(f"{ignored = }, {inserted = }")

ignored = 2, inserted = 1


In [6]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@old.com'),
 (3, 'cathy', 'cathy@old.com'),
 (4, 'dave', 'dave@new.com')]

Users 2 and 3 keep their old names and emails; dave is added.

## Insert or replace

Insert the new rows, and overwrite each existing row so it equals its record.

In [7]:
reset()
replaced, inserted = sqlite.insert_or_replace(engine, t_user, records)
print(f"{replaced = }, {inserted = }")

replaced = 2, inserted = 1


In [8]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'robert', 'bob@new.com'),
 (3, 'catherine', 'cathy@new.com'),
 (4, 'dave', 'dave@new.com')]

## Insert or merge

Insert the new rows; on existing rows update only the `columns` you name, and keep the rest. Here only `email` is updated.

In [9]:
reset()
updated, inserted = sqlite.insert_or_merge(engine, t_user, records, columns=["email"])
print(f"{updated = }, {inserted = }")

updated = 2, inserted = 1


In [10]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@new.com'),
 (3, 'cathy', 'cathy@new.com'),
 (4, 'dave', 'dave@new.com')]

Users 2 and 3 get their new emails, but keep their old names (bob and cathy, not robert and catherine). dave is inserted in full.

## Summary

| Strategy | Existing rows (2, 3) | New row (4) | Returns |
| :--- | :--- | :--- | :--- |
| `insert_or_ignore` | unchanged | inserted | `(ignored, inserted)` |
| `insert_or_replace` | overwritten with the record | inserted | `(replaced, inserted)` |
| `insert_or_merge` | only `columns` updated | inserted | `(updated, inserted)` |

The two counts always add up to the number of records. The same three functions exist for PostgreSQL and MySQL, under `api.postgres` and `api.mysql`.